# 实验：vLLM 单步 RAG 基线

> 学生版 notebook 只展示一个最小可运行链路：`search` 一次，然后把检索结果喂给模型生成答案。

> 不提供完整 agent loop。


## 0. 环境与服务说明

假设你已经在服务器上启动了 vLLM OpenAI-compatible 服务。

这个 notebook 不自己做模型推理服务，只调用已有 endpoint。


In [1]:
!python --version
!pip install -r agent/requirements.txt


Python 3.10.17
Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://mirrors.huaweicloud.com/repository/pypi/simple

[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## 1. vLLM 服务配置

这里先填好模型名和服务地址。

如果只是做单步 RAG，这里不需要 vLLM 开启自动工具调用。


In [3]:
from pathlib import Path
import sys

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

VLLM_BASE_URL = 'http://127.0.0.1:8000/v1'
# MODEL_NAME = 'pangu_auto'
MODEL_NAME = 'qwen_auto'
API_KEY = 'dummy'


## 2. 服务器上预构建 BM25 索引

正式 baseline 使用 `browsecomp-plus-corpus` 全量语料。

索引一般只构建一次；如果已经建好，可以直接跳过这一格。


In [4]:
corpus_path = str('browsecomp-plus-corpus')
bm25_index_path = str('indexes/browsecomp_plus_bm25.sqlite')

# 首次在服务器上执行；已经建过就跳过
!python -m agent.build_bm25_index --corpus-path ./browsecomp-plus-corpus --index-path ./indexes/browsecomp_plus_bm25.sqlite --overwrite


{
  "index_path": "indexes/browsecomp_plus_bm25.sqlite",
  "num_documents": 10000
}


## 3. 初始化检索与 vLLM 客户端


In [5]:
from agent.vllm_client import VLLMClient
from agent.tools import build_searcher, retrieve_once, format_rag_context
from agent.dataset_utils import load_jsonl

hard50_path = str(project_root / 'browsecomp_plus_hard50.jsonl')
output_path = str(project_root / 'runs' / 'student_rag_predictions.jsonl')

client = VLLMClient(base_url=VLLM_BASE_URL, api_key=API_KEY)
searcher = build_searcher(index_path=bm25_index_path)
print('search_type:', searcher.search_type)


search_type: bm25_sqlite_fts5


## 4. 单步 RAG 函数

流程很简单：

1. 对问题做一次 `search`
2. 把检索结果拼成上下文
3. 让模型基于这些上下文直接回答


In [6]:
def answer_with_single_search(question: str, top_k: int = 5, max_tokens: int = 4096):
    results = retrieve_once(searcher=searcher, query=question, k=top_k)
    context = format_rag_context(results)
    messages = [
        {
            'role': 'system',
            'content': (
                'You are a retrieval-augmented QA assistant. '
                'Use the provided documents to answer the question. '
                'If the documents are insufficient, say the evidence is insufficient. '
                'Please answer in the format:\n'
                'Explanation: <brief explanation>\n'
                'Exact Answer: <final answer>'
            ),
        },
        {
            'role': 'user',
            'content': f'Question:\n{question}\n\nRetrieved Documents:\n{context}',
        },
    ]
    response = client.simple_chat(
        model=MODEL_NAME,
        messages=messages,
        temperature=0.0,
        max_tokens=max_tokens,
    )
    return {
        'question': question,
        'retrieved_docs': results,
        'response': response,
        'answer_text': response['choices'][0]['message']['content'],
    }


## 5. 单条样本演示


In [8]:
rows = load_jsonl(hard50_path, limit=1)
demo_row = rows[0]
demo = answer_with_single_search(demo_row['query'])

print('query_id:', demo_row['query_id'])
print('gold_answer:', demo_row['answer'])
print('\nretrieved docids:', [doc['docid'] for doc in demo['retrieved_docs']])
print('\nmodel answer:\n')
print(demo['answer_text'])


query_id: 442
gold_answer: THE DAWN OF AUSTRALIAN COLONISATION

retrieved docids: ['41740', '18896', '30148', '61171', '89724']

model answer:

<think>
Okay, let's try to figure out the answer to this question. The user is asking for the title of the first chapter of the second book discussed. Let me break down the information given.

First, the book in question was first published in the 1920s. It deals with inland discoveries and was published by a company founded in the 1880s. There's a description of a barrel-shaped floating vessel on pages 332-339. Then, on pages 463-464, there's a spear attack on a botanist's party. The botanist returned to London in the 1830s. The author got married in the 1890s. Another book by the author was published between 1900-1910, discussing the groundwork for early affluence of long-standing settlements.

Looking at the retrieved documents, Document 2 and 3 are about "The Coming of the British to Australia 1788 to 1829" and "Early Explorers in Australia

## 6. 生成轨迹文件（submission.jsonl）

这里将单步 RAG 的结果整理为统一格式的轨迹文件，每条记录包含完整的 `messages` 对话历史。



In [9]:
import json
from pathlib import Path


def build_trajectory(question: str, top_k: int = 5, max_tokens: int = 512) -> dict:
    """
    执行单步 RAG 并返回符合提交格式的轨迹记录。
    
    返回的 record 包含：
    - query_id, status, predicted_answer
    - messages: [system, user, assistant(tool_calls), tool, assistant(final)]
    """
    results = retrieve_once(searcher=searcher, query=question, k=top_k)
    context = format_rag_context(results)

    # 构建 search 工具调用参数（用于轨迹记录）
    search_args = json.dumps({"query": question}, ensure_ascii=False)
    # 构建工具返回结果
    tool_result = json.dumps(
        [
            {
                "docid": doc["docid"],
                "score": doc["score"],
                "snippet": doc["snippet"],
                "url": doc.get("url", ""),
            }
            for doc in results
        ],
        ensure_ascii=False,
    )

    # 构建完整的 messages 轨迹
    messages = [
        {
            "role": "system",
            "content": (
                "You are a Deep Research Agent. "
                "Your task is to answer complex questions by searching a document corpus. "
                "Use the provided search results to find evidence and give a final answer. "
                "Please answer in the format:\n"
                "Explanation: <brief explanation>\n"
                "Exact Answer: <final answer>\n"
                "Confidence: <percentage>%"
            ),
        },
        {
            "role": "user",
            "content": question,
        },
        {
            "role": "assistant",
            "content": "",
            "tool_calls": [
                {
                    "id": "call_1",
                    "type": "function",
                    "function": {
                        "name": "search",
                        "arguments": search_args,
                    },
                }
            ],
        },
        {
            "role": "tool",
            "tool_call_id": "call_1",
            "content": tool_result,
        },
    ]

    # 调用模型生成最终答案
    rag_messages = [
        {
            "role": "system",
            "content": (
                "You are a retrieval-augmented QA assistant. "
                "Use the provided documents to answer the question. "
                "If the documents are insufficient, say the evidence is insufficient. "
                "Please answer in the format:\n"
                "Explanation: <brief explanation>\n"
                "Exact Answer: <final answer>"
            ),
        },
        {
            "role": "user",
            "content": f"Question:\n{question}\n\nRetrieved Documents:\n{context}",
        },
    ]
    response = client.simple_chat(
        model=MODEL_NAME,
        messages=rag_messages,
        temperature=0.0,
        max_tokens=max_tokens,
    )
    answer_text = response["choices"][0]["message"]["content"]

    # 追加最终 assistant 回复
    messages.append({
        "role": "assistant",
        "content": answer_text,
    })

    return {
        "query_id": None,  # 由外部填充
        "status": "completed",
        "predicted_answer": answer_text,
        "messages": messages,
    }


def generate_submission(rows, output_path: str, top_k: int = 5, max_tokens: int = 512):
    """批量生成轨迹并写入 submission.jsonl。"""
    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)

    records = []
    with output_file.open("w", encoding="utf-8") as fout:
        for i, row in enumerate(rows):
            print(f"[{i+1}/{len(rows)}] Processing query_id={row['query_id']}...")
            traj = build_trajectory(row["query"], top_k=top_k, max_tokens=max_tokens)
            traj["query_id"] = row["query_id"]
            records.append(traj)
            json.dump(traj, fout, ensure_ascii=False)
            fout.write("\n")

    print(f"\nSaved {len(records)} trajectories to: {output_path}")
    return records


# 生成轨迹文件
submission_path = str(project_root / "runs" / "submission.jsonl")
rows = load_jsonl(hard50_path, limit=50)  # 处理全部 50 条
records = generate_submission(rows, output_path=submission_path)

# 打印第一条样本确认格式
print("\n--- 第一条轨迹样例 ---")
sample = records[0]
print(f"query_id: {sample['query_id']}")
print(f"status: {sample['status']}")
print(f"predicted_answer (前200字): {sample['predicted_answer'][:200]}...")
print(f"messages 条数: {len(sample['messages'])}")
for msg in sample["messages"]:
    role = msg["role"]
    has_tool_calls = "tool_calls" in msg
    content_preview = str(msg.get("content", ""))[:80]
    print(f"  [{role}] tool_calls={has_tool_calls} | {content_preview}...")


[1/50] Processing query_id=442...
[2/50] Processing query_id=549...
[3/50] Processing query_id=26...
[4/50] Processing query_id=471...
[5/50] Processing query_id=761...
[6/50] Processing query_id=815...
[7/50] Processing query_id=216...
[8/50] Processing query_id=314...
[9/50] Processing query_id=241...
[10/50] Processing query_id=930...
[11/50] Processing query_id=651...
[12/50] Processing query_id=1082...
[13/50] Processing query_id=159...
[14/50] Processing query_id=5...
[15/50] Processing query_id=395...
[16/50] Processing query_id=944...
[17/50] Processing query_id=747...
[18/50] Processing query_id=635...
[19/50] Processing query_id=113...
[20/50] Processing query_id=556...
[21/50] Processing query_id=153...
[22/50] Processing query_id=263...
[23/50] Processing query_id=827...
[24/50] Processing query_id=730...
[25/50] Processing query_id=79...
[26/50] Processing query_id=86...
[27/50] Processing query_id=397...
[28/50] Processing query_id=427...
[29/50] Processing query_id=1072.

## 7. 自动评估

使用 `agent/eval.py` 中的评估函数，调用 LLM 自动判断预测答案与标准答案是否一致，并输出准确率和详细评估结果。

评估模型可以使用 Qwen 或 Pangu，这里以 Qwen 为例。



In [10]:
from agent.eval import run_evaluation

# 使用 Qwen3-8B 作为评估模型（也可以换成 openPangu-Embedded-7B-DeepDiver）
EVAL_MODEL = MODEL_NAME

eval_output_path = str("runs/eval_results.jsonl")

summary, details = run_evaluation(
    submission_path=submission_path,
    dataset_path=hard50_path,
    model_name=EVAL_MODEL,
    base_url=VLLM_BASE_URL,
    api_key=API_KEY,
    output_path=eval_output_path,
    verbose=True,
)

print(f"\n{'='*50}")
print(f"📊 评估摘要")
print(f"{'='*50}")
print(f"总题目数:      {summary['total_queries']}")
print(f"正确数:        {summary['correct']}")
print(f"错误数:        {summary['incorrect']}")
print(f"准确率 (ACC):  {summary['accuracy']:.2%}")
print(f"平均工具调用:  {summary['avg_tool_calls_per_query']}")
print(f"平均检索文档:  {summary['avg_retrieved_docs_per_query']}")
print(f"评估模型:      {summary['eval_model']}")

# 打印几个错误案例
print(f"\n{'='*50}")
print("🔍 错误案例 (前 5 条)")
print(f"{'='*50}")
error_cases = [d for d in details if d["eval_judgment"] == "INCORRECT"]
for case in error_cases[:5]:
    print(f"\nquery_id: {case['query_id']}")
    print(f"  Gold:      {case['gold_answer'][:100]}")
    print(f"  Predicted: {case['predicted_answer'][:100]}")
    print(f"  Reasoning: {case['eval_reasoning'][:150]}")


[442] INCORRECT | pred=<think>
Okay, let's try to figure out the answer to this que...
[549] INCORRECT | pred=<think>
Okay, let's try to figure out this question. The use...
[26] INCORRECT | pred=<think>
Okay, let's try to figure out the name of the club b...
[471] INCORRECT | pred=<think>
Okay, let's tackle this question step by step. The u...
[761] INCORRECT | pred=<think>
Okay, let's tackle this question. The user is asking...
[815] INCORRECT | pred=<think>
Okay, let's tackle this question step by step. The u...
[216] INCORRECT | pred=<think>
Okay, let's try to figure out the author of the book...
[314] INCORRECT | pred=<think>
Okay, let's tackle this question step by step. The u...
[241] INCORRECT | pred=<think>
Okay, let's tackle this question step by step. The u...
[930] INCORRECT | pred=<think>
Okay, let's try to figure out the answer to this que...
[651] INCORRECT | pred=<think>
Okay, let's try to figure out the name of the public...
[1082] INCORRECT | pred=<think>
Okay, let's 